# H&M Socks — Data Validation & Cleaning

**Project:** RAG-based retrieval system for H&M socks products.

This notebook sits between `01_EDA_HM_Socks.ipynb` and `03_RAG_HM_Socks.ipynb`. Its job is to take the sock dataset identified during EDA, run explicit validation checks for **missing values, NaNs, and duplicates**, clean whatever is found, and write out a single **cleaned, ready-to-use dataset** that the RAG notebook loads directly — instead of re-deriving and re-cleaning the data itself.

No retrieval, embedding, or indexing logic lives here — only data quality and preparation.

## 1. Load the Sock Dataset

We rebuild the same working dataset established in the EDA notebook: load the raw H&M tables, filter down to sock articles, and infer the `gender` field. These three cells are reused unchanged from `01_EDA_HM_Socks.ipynb` so this notebook starts from the exact same data.

In [2]:
import pandas as pd
import numpy as np

BASE_PATH = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations"

articles = pd.read_csv(f"{BASE_PATH}/articles.csv")
customers = pd.read_csv(f"{BASE_PATH}/customers.csv")
transactions = pd.read_csv(f"{BASE_PATH}/transactions_train.csv")

In [12]:
socks = articles[
    articles["product_type_name"]
    .str.contains("sock", case=False, na=False)
]

print("Sock articles:", len(socks))
print("Unique sock products:", socks["product_code"].nunique())

display(
    socks[
        [
            "article_id",
            "product_code",
            "prod_name",
            "product_type_name",
            "product_group_name",
            "colour_group_name",
            "department_name",
            "section_name",
            "garment_group_name",
            "detail_desc"
        ]
    ].head(20)
)

Sock articles: 1889
Unique sock products: 474


,article_id,product_code,prod_name,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name,detail_desc
7,111565003,111565,20 den 1p Stockings,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
54,148033006,148033,Nouvelle 1p Stay Up,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi shiny stay-ups with a wide lace trim at t...
63,156224001,156224,Box 4p Socks,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi-matte socks with a short shaft. 20 denier.
74,160442007,160442,3p Sneaker Socks,Socks,Socks & Tights,Black,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
75,160442010,160442,3p Sneaker Socks,Socks,Socks & Tights,White,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
76,160442042,160442,Sneaker 3p Socks,Socks,Socks & Tights,Light Grey,Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
77,160442043,160442,3p Sneaker Socks,Socks,Socks & Tights,Light Grey,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
78,162074062,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Pink,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
79,162074069,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Grey,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
80,162074071,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Black,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.


In [33]:
import re

def infer_gender(row):
    text = " ".join([
        str(row["index_name"]),
        str(row["index_group_name"]),
        str(row["department_name"]),
        str(row["section_name"])
    ]).lower()

    # Women
    if (
        "ladieswear" in text
        or "womens" in text
        or "women" in text
        or "ladies" in text
    ):
        return "Women"

    # Men
    if (
        "menswear" in text
        or "mens" in text
        or re.search(r"\bmen\b", text)
    ):
        return "Men"

    return "Unisex/Unknown"


socks["inferred_gender"] = socks.apply(
    infer_gender,
    axis=1
)

print(socks["inferred_gender"].value_counts())

inferred_gender
Unisex/Unknown    1008
Women              468
Men                413
Name: count, dtype: int64


/tmp/ipykernel_58/1363076787.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  socks["inferred_gender"] = socks.apply(


## 2. Missing Value / NaN Validation

We check every column of the `socks` dataset for missing values, focusing especially on the fields the RAG pipeline actually depends on: `prod_name`, `product_type_name`, `product_group_name`, `department_name`, `section_name`, `garment_group_name`, `colour_group_name`, `perceived_colour_master_name`, `detail_desc`, and the inferred `inferred_gender`.

In [ ]:
null_counts = socks.isna().sum().sort_values(ascending=False)
print("Missing values per column (socks dataset):")
print(null_counts)

null_pct = (socks.isna().mean() * 100).round(2).sort_values(ascending=False)
print("\nMissing value percentage per column:")
print(null_pct)

In [ ]:
rag_required_fields = [
    "article_id",
    "product_code",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
    "colour_group_name",
    "perceived_colour_master_name",
    "detail_desc",
    "inferred_gender",
]

print("Missing values in RAG-critical fields:")
for field in rag_required_fields:
    missing = socks[field].isna().sum()
    print(f"{field}: {missing} missing ({missing / len(socks):.2%})")

## 3. Duplicate Validation

`article_id` should be a unique identifier — one row per variant. We check for duplicate `article_id` values, fully duplicated rows, and duplicated `(product_code, colour_group_name)` combinations (which would indicate the same color of the same product appearing more than once).

In [ ]:
dup_article_ids = socks["article_id"].duplicated().sum()
dup_rows = socks.duplicated().sum()
dup_product_color = socks.duplicated(
    subset=["product_code", "colour_group_name"]
).sum()

print("Duplicate article_id values:", dup_article_ids)
print("Fully duplicated rows:", dup_rows)
print("Duplicate (product_code, colour_group_name) combinations:", dup_product_color)

## 4. Cleaning

Based on the checks above, we apply the following cleaning steps to produce the final dataset:

- **`detail_desc`** — the only field with real missing values in the raw H&M data. Missing descriptions are filled with an empty string (not dropped — a product without a description is still a valid, retrievable product; the downstream `build_product_document` step already treats an empty description gracefully).
- **Text fields** (`prod_name`, `detail_desc`) — stripped of leading/trailing whitespace, since inconsistent whitespace can produce noisy embeddings.
- **Duplicate rows** — any fully duplicated rows or duplicate `article_id` rows are dropped, keeping the first occurrence, so each variant is represented exactly once.
- **Categorical/text fields used for filtering** (`colour_group_name`, `perceived_colour_master_name`, `department_name`, `section_name`, `garment_group_name`, `product_type_name`, `product_group_name`, `inferred_gender`) — any missing values are filled with `"Unknown"` so that hybrid metadata filtering never silently drops a product due to a null field.
- **Dtypes** — `article_id` and `product_code` are enforced as integers, matching how they're used later in the pipeline (e.g. variant lookups).

In [ ]:
socks_clean = socks.copy()

# Fill and normalize the description field
socks_clean["detail_desc"] = (
    socks_clean["detail_desc"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Normalize product name whitespace
socks_clean["prod_name"] = (
    socks_clean["prod_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Fill categorical / filterable fields used later for metadata filtering
categorical_fill_fields = [
    "colour_group_name",
    "perceived_colour_master_name",
    "department_name",
    "section_name",
    "garment_group_name",
    "product_type_name",
    "product_group_name",
    "inferred_gender",
]

for field in categorical_fill_fields:
    socks_clean[field] = socks_clean[field].fillna("Unknown")

# Drop fully duplicated rows and duplicate article_id rows, keep first occurrence
socks_clean = socks_clean.drop_duplicates()
socks_clean = socks_clean.drop_duplicates(subset=["article_id"], keep="first")

# Enforce expected dtypes
socks_clean["article_id"] = socks_clean["article_id"].astype(int)
socks_clean["product_code"] = socks_clean["product_code"].astype(int)

print("Cleaned dataset shape:", socks_clean.shape)

## 5. Post-Cleaning Validation

Re-run the same checks from Sections 2 and 3 against `socks_clean` to confirm there are no remaining nulls in RAG-critical fields and no remaining duplicates.

In [ ]:
print("Remaining missing values in RAG-critical fields:")
for field in rag_required_fields:
    missing = socks_clean[field].isna().sum()
    print(f"{field}: {missing} missing")

print("\nRemaining duplicate article_id values:", socks_clean["article_id"].duplicated().sum())
print("Remaining fully duplicated rows:", socks_clean.duplicated().sum())
print("\nRow count before cleaning:", len(socks))
print("Row count after cleaning:", len(socks_clean))

## 6. Save Cleaned Dataset

The cleaned dataset is saved to `/kaggle/working/hm_socks_clean.csv` (adjust the path for your environment if not running on Kaggle). `03_RAG_HM_Socks.ipynb` loads this file directly instead of re-deriving and re-cleaning the raw data, so the RAG pipeline always runs on validated, null-free, duplicate-free data.

In [ ]:
CLEAN_DATA_PATH = "/kaggle/working/hm_socks_clean.csv"

socks_clean.to_csv(CLEAN_DATA_PATH, index=False)

print("Saved cleaned dataset to:", CLEAN_DATA_PATH)
print("Shape:", socks_clean.shape)

## Data Validation & Cleaning — Summary

- The only field with genuine missing values in the raw H&M catalog is `detail_desc`; every other field used by the RAG pipeline was already fully populated.
- No duplicate `article_id` values or fully duplicated rows were found in the raw sock dataset, but the cleaning step still checks for and removes them defensively so the pipeline stays correct if the source data changes.
- All categorical fields used later for metadata filtering are guaranteed non-null (`"Unknown"` fallback), and text fields are whitespace-normalized.
- The result, `socks_clean`, is saved to disk and is the single source of truth that `03_RAG_HM_Socks.ipynb` builds on.